In [ ]:
https://en.wikipedia.org/wiki/Outline_of_the_Solar_System
https://en.wikipedia.org/wiki/Lists_of_astronomical_objects

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
from typing import List, Dict, Tuple, Optional, Any
import re
import copy

In [ ]:
def _extract_citations(element) -> List[int]:
    """Extract citation numbers from sup tags (e.g., [1], [2])."""
    citations = []
    for sup in element.find_all('sup'):
        cls = sup.get('class', [])
        if 'reference' in cls or 'cite_ref' in cls:
            text = sup.get_text(strip=True)
            # Try to extract number from [1], [2], etc.
            match = re.search(r'\[(\d+)\]', text)
            if match:
                citations.append(int(match.group(1)))
    return citations


def getTable(table, base_url: str) -> pd.DataFrame:
    """Parse a Wikipedia table into a DataFrame with text, links, and citations for each column."""
    rows = table.find_all('tr')
    if not rows:
        return pd.DataFrame()
    
    # Extract header row (look for th cells, or use first row)
    header_row = None
    header_cells = None
    data_start_idx = 0
    
    for idx, row in enumerate(rows):
        ths = row.find_all('th')
        if ths:
            header_row = row
            header_cells = ths
            data_start_idx = idx + 1
            break
    
    # If no header row found, use first row as header
    if header_cells is None:
        first_row = rows[0]
        tds = first_row.find_all('td')
        if tds:
            header_cells = tds
            data_start_idx = 1
        else:
            return pd.DataFrame()
    
    # Extract column names
    col_names = [cell.get_text(strip=True) for cell in header_cells]
    
    # Process data rows
    data_rows = []
    for row_idx in range(data_start_idx, len(rows)):
        row = rows[row_idx]
        cells = row.find_all(['td', 'th'])
        
        if not cells:
            continue
        
        row_data = {}
        for col_idx, cell in enumerate(cells):
            if col_idx >= len(col_names):
                break
            
            col_name = col_names[col_idx]
            
            # Make a copy to avoid modifying original
            cell_copy = copy.copy(cell)
            
            # Extract citations before removing them
            citations = _extract_citations(cell_copy)
            
            # Remove citation sup tags
            for sup in cell_copy.find_all('sup'):
                sup.decompose()
            
            # Extract links
            links = []
            for a in cell_copy.find_all('a', href=True):
                link_text = a.get_text(strip=True)
                link_href = a.get('href', '')
                if link_text and link_href:
                    full_url = urljoin(base_url, link_href)
                    links.append((link_text, full_url))
            
            # Get text
            text = cell_copy.get_text(separator=' ', strip=True)
            
            # Remove parenthetical content (usually conversions/units)
            # text = re.sub(r'\s*\([^)]*\)', '', text).strip()
            
            # Store the three variants
            row_data[f"{col_name}_text"] = text if text else None
            row_data[f"{col_name}_links"] = links if links else []
            row_data[f"{col_name}_citations"] = citations if citations else []
        
        data_rows.append(row_data)
    
    if not data_rows:
        return pd.DataFrame()
    
    return pd.DataFrame(data_rows)


def getInfoBox(soup, baseurl):
    table = soup.select_one('table.infobox')
    rows = table.find_all('tr')

    data = []
    current_header = "General"

    for row in rows:
        # Check if this row is an infobox header
        header_cell = row.find('th', class_='infobox-header')
        if header_cell:
            current_header = header_cell.get_text(strip=True)
            continue
        
        th = row.find('th')
        td = row.find('td', class_='infobox-data')
        
        if th and td:
            label = th.get_text(strip=True)
            
            list_items = td.find_all('li')
            containers = list_items if list_items else [td]
            
            values_list = []
            links_list = []
            citations_list = []
            
            for container_orig in containers:
                # Make a deep copy to avoid destroying citations in the original soup object
                # if this cell is executed multiple times!
                container = copy.copy(container_orig)
                
                # Extract citations: Look for sup tags that contain bracketed numbers/letters or have class reference
                cites = []
                for sup in container.find_all('sup'):
                    cls = sup.get('class', [])
                    if 'reference' in cls or 'cite_ref' in cls or (sup.get_text() and re.match(r'^\[\w+\]$', sup.get_text().strip())):
                        cites.append(sup.get_text(strip=True))
                        sup.decompose()
                
                # Extract links, ignoring units (heuristic: ignoring short strings or numbers)
                lnks = []
                for a in container.find_all('a'):
                    link_text = a.get_text(strip=True)
                    link_href = a.get('href', '')
                    # Simple heuristic to exclude typical unit/symbol links
                    if len(link_text) > 2 and not any(char.isdigit() for char in link_text):
                        if link_href:
                            full_url = urljoin(baseurl, link_href)
                            lnks.append(full_url)
                
                # Re-get the value after decomposing citations in the temporary copy
                # Also remove parenthetical text which usually contains redundant conversions/units
                value_clean = container.get_text(separator=' ', strip=True)
                #value_clean = re.sub(r'\s*\([^)]*\)', '', value_clean).strip()
                
                # If after stripping units it becomes empty, we might want to skip it, 
                # but we shouldn't lose the citations if it had any.
                # We can merge its citations into the previous item in the list.
                if not value_clean:
                    if cites and citations_list:
                        # Append these citations to the previous item's citations
                        prev_cites = citations_list[-1]
                        if prev_cites:
                            citations_list[-1] = prev_cites + ", " + ", ".join(cites)
                        else:
                            citations_list[-1] = ", ".join(cites)
                    continue
                    
                values_list.append(value_clean)
                links_list.append(", ".join(lnks) if lnks else None)
                citations_list.append(", ".join(cites) if cites else None)
                
            if not values_list:
                continue
            
            value_out = values_list if list_items else (values_list[0] if values_list else None)
            links_out = links_list if list_items else (links_list[0] if links_list else None)
            cites_out = citations_list if list_items else (citations_list[0] if citations_list else None)
            
            # Collect links from the row label (<th>) as a list of absolute URLs
            label_links = [
                urljoin(baseurl, a.get('href'))
                for a in th.find_all('a', href=True)
                if a.get('href')
            ]
            # Optional de-dup while preserving order
            label_links = list(dict.fromkeys(label_links))

            data.append({
                'Metadata': current_header,
                'Label': label,
                'Label Links': label_links,
                'Value': value_out,
                'Links': links_out,
                'Citations': cites_out
            })

    df = pd.DataFrame(data)
    df.set_index(['Metadata', 'Label'], inplace=True)
    return df

In [ ]:
def fetch_wikipedia_body(url: str, timeout: int = 20):
    """Fetch a Wikipedia page and return parsed soup + body content container."""
    response = requests.get(url, timeout=timeout, headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    })
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Prefer id=bodyContent, fallback to class=mw-body-content
    body = soup.find(id="bodyContent")
    if body is None:
        body = soup.find(class_="mw-body-content")
    if body is None:
        raise ValueError("Could not find Wikipedia body content (id=bodyContent or class=mw-body-content).")

    return soup, body


def _alpha_index(n: int, upper: bool = False) -> str:
    """Convert 1-based index to alphabetic labels: 1->a, 27->aa."""
    chars = []
    while n > 0:
        n, rem = divmod(n - 1, 26)
        chars.append(chr(ord("A" if upper else "a") + rem))
    return "".join(reversed(chars))


def _to_roman(n: int) -> str:
    """Convert integer to Roman numeral (supports practical ordered-list sizes)."""
    numerals = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")
    ]
    result = []
    value = n
    for arabic, roman in numerals:
        while value >= arabic:
            result.append(roman)
            value -= arabic
    return "".join(result)


def _detect_order_style(ol) -> str:
    """Detect ordered-list marker style from type/style/class, defaulting to decimal."""
    explicit_type = ol.get("type")
    if explicit_type in {"1", "a", "A", "i", "I"}:
        return explicit_type

    style_blob = " ".join([
        (ol.get("style") or ""),
        " ".join(ol.get("class", []))
    ]).lower()

    if "lower-alpha" in style_blob or "lower-latin" in style_blob:
        return "a"
    if "upper-alpha" in style_blob or "upper-latin" in style_blob:
        return "A"
    if "lower-roman" in style_blob:
        return "i"
    if "upper-roman" in style_blob:
        return "I"
    return "1"


def _ordered_key(index: int, style: str) -> str:
    if style == "1":
        return str(index)
    if style == "a":
        return _alpha_index(index, upper=False)
    if style == "A":
        return _alpha_index(index, upper=True)
    if style == "i":
        return _to_roman(index).lower()
    if style == "I":
        return _to_roman(index).upper()
    return str(index)


def parse_body_to_dataframe(base_url: str, body) -> pd.DataFrame:
    """Parse headers, paragraphs, lists, and tables from body content into a pandas DataFrame."""
    rows: List[Dict[str, Any]] = []
    heading_state: Dict[str, Optional[str]] = {f"h{i}": None for i in range(1, 7)}
    list_counter = 0
    table_counter = 0

    # Preserve document order while only handling headers, paragraphs, lists, and tables
    for element in body.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "ul", "ol", "table"]):
        tag = element.name.lower()

        if tag.startswith("h"):
            level = int(tag[1])
            heading_text = element.get_text(" ", strip=True)
            if not heading_text:
                continue

            heading_state[f"h{level}"] = heading_text
            for deeper in range(level + 1, 7):
                heading_state[f"h{deeper}"] = None
            continue

        links: List[Tuple[str, str]] = []
        for a in element.find_all("a", href=True):
            link_text = a.get_text(" ", strip=True)
            link_url = urljoin(base_url, a["href"])
            links.append((link_text, link_url))

        row = {f"h{i}": heading_state[f"h{i}"] for i in range(1, 7)}
        row["listID"] = None
        row["tableID"] = None
        row["text"] = None
        row["list"] = None
        row["links"] = links

        if tag == "p":
            text = element.get_text(" ", strip=True)
            if not text:
                continue
            row["text"] = text
            rows.append(row)
            continue

        if tag == "ul":
            items = [li.get_text(" ", strip=True) for li in element.find_all("li", recursive=False)]
            items = [item for item in items if item]
            if not items:
                continue

            list_counter += 1
            row["listID"] = f"{list_counter}"
            row["list"] = items
            rows.append(row)
            continue

        if tag == "ol":
            items = [li.get_text(" ", strip=True) for li in element.find_all("li", recursive=False)]
            items = [item for item in items if item]
            if not items:
                continue

            style = _detect_order_style(element)
            ordered_items = {_ordered_key(i + 1, style): item for i, item in enumerate(items)}

            list_counter += 1
            row["listID"] = f"{list_counter}"
            row["list"] = ordered_items
            rows.append(row)
            continue

        if tag == "table":
            # Skip infobox tables
            if element.get("class") and "infobox" in element.get("class", []):
                continue
            
            table_df = getTable(element, base_url)
            if table_df.empty:
                continue
            
            table_counter += 1
            table_id = f"{table_counter}"
            
            # Add heading context and table ID to each row from the table
            for _, table_row in table_df.iterrows():
                context_row = {f"h{i}": heading_state[f"h{i}"] for i in range(1, 7)}
                context_row["listID"] = None
                context_row["tableID"] = table_id
                context_row["text"] = None
                context_row["list"] = None
                context_row["links"] = []
                
                # Merge table row data into context row
                context_row.update(table_row.to_dict())
                rows.append(context_row)
            continue

    # Build dynamic columns: h1-h6, listID, tableID, text, list, links, plus all table columns
    # First, collect all unique table column names
    table_cols = set()
    for row in rows:
        for key in row:
            if key.endswith("_text") or key.endswith("_links") or key.endswith("_citations"):
                table_cols.add(key)
    
    table_cols = sorted(list(table_cols))
    columns = ["h1", "h2", "h3", "h4", "h5", "h6", "listID", "tableID", "text", "list", "links"] + table_cols
    
    # Ensure all rows have all columns
    for row in rows:
        for col in columns:
            if col not in row:
                row[col] = None
    
    return pd.DataFrame(rows, columns=columns)


def scrape_wikipedia_to_dataframe(url: str) -> pd.DataFrame:
    """Convenience wrapper to fetch + parse into DataFrame."""
    _, body = fetch_wikipedia_body(url)
    return parse_body_to_dataframe(url, body)

In [6]:
# Example usage
url = "https://en.wikipedia.org/wiki/Solar_System"
df = scrape_wikipedia_to_dataframe(url)
print(df.shape)
df.head(10)

(267, 10)


,h1,h2,h3,h4,h5,h6,listID,text,list,links
0,None,None,None,None,None,None,1,None,"[Local Interstellar Cloud, Local Bubble [ 1 ],...","[(Local Interstellar Cloud, https://en.wikiped..."
1,None,None,None,None,None,None,2,None,"[Proxima Centauri, (4.2465 ly ) [ D 1 ], Alpha...","[(Proxima Centauri, https://en.wikipedia.org/w..."
2,None,None,None,None,None,None,3,None,"[Mercury, Venus, Earth, Mars, Jupiter, Saturn,...","[(Mercury, https://en.wikipedia.org/wiki/Mercu..."
3,None,None,None,None,None,None,4,None,"[Ceres, Orcus, Pluto, Haumea, Quaoar, Makemake...","[(Ceres, https://en.wikipedia.org/wiki/Ceres_(..."
4,None,None,None,None,None,None,None,The Solar System is the gravitationally bound ...,None,"[(gravitationally, https://en.wikipedia.org/wi..."
5,None,None,None,None,None,None,None,The Sun accounts for 99.86% of the Solar Syste...,None,"[(Sun's core, https://en.wikipedia.org/wiki/So..."
6,None,None,None,None,None,None,None,The next most massive objects of the system ar...,None,"[(most massive objects of the system, https://..."
7,None,None,None,None,None,None,None,Objects of planetary mass that do not dominate...,None,"[(Objects of planetary mass, https://en.wikipe..."
8,None,None,None,None,None,None,None,Many objects in the Solar System do not orbit ...,None,"[(natural satellites, https://en.wikipedia.org..."
9,None,None,None,None,None,None,None,"Within the heliosphere, the Solar System is co...",None,"[(plasma, https://en.wikipedia.org/wiki/Plasma..."


In [ ]:
# Preview parsed lists and tables
print("Lists:")
print(df[df["listID"].notna()][["h1", "h2", "h3", "listID", "list"]].head(10))
print("\nTables:")
table_df = df[df["tableID"].notna()]
if not table_df.empty:
    # Show only heading context + table ID + first few table columns
    table_cols = [c for c in df.columns if c.endswith("_text") or c.endswith("_links")]
    print(table_df[["h1", "h2", "tableID"] + table_cols[:6]].head(10))

,h1,h2,h3,listID,list,links
247,None,External links,None,158,"[Arabic, Chinese]","[(Arabic, https://en.wikipedia.org/wiki/List_o..."
248,None,External links,None,159,"[Most massive, Highest temperature, Lowest tem...","[(Most massive, https://en.wikipedia.org/wiki/..."
249,None,External links,None,160,[bright],"[(bright, https://en.wikipedia.org/wiki/List_o..."
250,None,External links,None,161,"[Candidates, Remnants]","[(Candidates, https://en.wikipedia.org/wiki/Li..."
251,None,External links,None,162,[Substellar object Brown dwarf Desert Sub Plan...,"[(Substellar object, https://en.wikipedia.org/..."
252,None,External links,None,163,"[Brown dwarf Desert Sub, Planet]","[(Brown dwarf, https://en.wikipedia.org/wiki/B..."
253,None,External links,None,164,"[Desert, Sub]","[(Desert, https://en.wikipedia.org/wiki/Brown-..."
254,None,External links,None,165,"[Category, Stars portal]","[(Category, https://en.wikipedia.org/wiki/Cate..."
255,None,External links,None,166,"[v, t, e]","[(v, https://en.wikipedia.org/wiki/Template:So..."
256,None,External links,None,167,"[Antikythera mechanism, Armillary sphere, Astr...","[(Antikythera mechanism, https://en.wikipedia...."
